In [1]:
from pyspark.sql import SparkSession
from delta import *

In [2]:
builder = SparkSession.builder.appName("Local_Lakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.sql.caseSensitive", "true")

In [3]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/11 15:18:47 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/11 15:18:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harsh/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3017f047-5fcc-47ef-a0e6-9799c3331612;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache

In [4]:
getPosts_df=spark.read.format("json")\
                 .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPosts/posts_results.jsonl")

In [5]:
getPosts_df.show(n=10,truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [7]:
def like_to_comment_ratio(num_likes,num_comments):
    if(num_comments==0):
        return 0.0
    return num_likes/num_comments

In [8]:
like_to_comment_ratio_udf=udf(like_to_comment_ratio,FloatType())

In [9]:
controversialPosts_df=getPosts_df.withColumn("like_to_comment_ratio",like_to_comment_ratio_udf(col("likeCount"),col("replyCount")))

In [10]:
controversialPosts_df.show(n=10,truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
controversialPosts_df=controversialPosts_df.filter((col("like_to_comment_ratio")!=0)&(col("replyCount")>=50))

In [12]:
controversialPosts_df.show(n=10,truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-----------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [13]:
controversialPosts_df=controversialPosts_df.sort(col("like_to_comment_ratio").asc())

In [14]:
controversialPosts_df.show(n=10,truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+-----------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
controversialPosts_df=controversialPosts_df.withColumn("splitted_text",array_distinct(split(lower(col("record.text")),r"\s+"))).select(col("splitted_text"),col("like_to_comment_ratio"))

In [18]:
controversialPosts_df=controversialPosts_df.withColumn("splitted_text",explode(col("splitted_text"))).select(col("splitted_text"),col("like_to_comment_ratio"))

In [19]:
controversialPosts_df=controversialPosts_df.groupBy(col("splitted_text")).agg(avg(col("like_to_comment_ratio")))

In [23]:
controversialPosts_df=controversialPosts_df.withColumnRenamed("avg(like_to_comment_ratio)","average_like_to_comment_ratio")

In [24]:
controversialPosts_df.sort(col("average_like_to_comment_ratio")).show(n=100,truncate=False)

+-------------------------------+-----------------------------+
|splitted_text                  |average_like_to_comment_ratio|
+-------------------------------+-----------------------------+
|heartlight.org                 |0.007299270015209913         |
|undertaking                    |0.007299270015209913         |
|"straight                      |0.007299270015209913         |
|finished:                      |0.007299270015209913         |
|through"                       |0.007299270015209913         |
|commentary.                    |0.007299270015209913         |
|scaling                        |0.008130080997943878         |
|✨fics                          |0.008130080997943878         |
|hollanov                       |0.008130080997943878         |
|li✨                            |0.008130080997943878         |
|saas                           |0.008130080997943878         |
|into:                          |0.008130080997943878         |
|“cool”                         |0.00813

In [ ]:
controversialPosts_df.write.format("parquet")\
                           .mode("append")\
                           .save("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/controversialPosts")